## Classifies Culpa Reviews by A 4-quandrant Political Compass

In [2]:
# Dependencies/libraries
import sys
import torch
import os
import pandas as pd
from transformers import pipeline

In [ ]:
# Check import success! Just for personal stuff
print(sys.version)
print(torch.__version__)

3.13.5 (main, Jun 11 2025, 15:36:57) [Clang 16.0.0 (clang-1600.0.26.6)]
2.14.0


In [ ]:
# This analysis involves a classifier for each axis
# X-axis model
focus_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Y-axis model
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

In [ ]:
# Read in the review sample csv 
cwd = os.getcwd()
csv_path = os.path.join(cwd, 'review.csv')

df = pd.read_csv(csv_path)

# Drop the unecessary information 
df = df.drop(columns = ['5',
                        '3',
                        'Unnamed: 3',
                        'museum project, midterm, final.',
                        '2000-01-01 00:00:00'])

df.loc[-1] = df.columns.to_list()
df.index = df.index + 1

df = df.sort_index()
df = df.rename(columns={df.columns[0]: "review", df.columns[1]: "rating"})
df.head()

In [ ]:
# Classify the focus (whether the review mainly addresses the quality of the professor or course) via zero-shot classification
candidate_labels = [
    "the review evaluates the professor: mastery/knowledge of the material, instructor, teaching style, clarity, personality, helpfulness, or behavior",
    "the review evaluates the course: workload, exams, quizzes, assignments, readings, grading, difficulty, or requirements"
]

def classify_focus(review):
    review = str(review)

    result = focus_classifier(
        review[:1000],
        candidate_labels=candidate_labels,
        hypothesis_template="This student review is mainly about {}."
    )

    scores = dict(zip(result["labels"], result["scores"]))

    professor_prob = scores[candidate_labels[0]]
    course_prob = scores[candidate_labels[1]]

    focus_score = course_prob - professor_prob

    if focus_score < -0.20:
        label = "professor-focused"
    elif focus_score > 0.20:
        label = "course-focused"
    else:
        label = "mixed"

    return pd.Series({
        "professor_prob": professor_prob,
        "course_prob": course_prob,
        "focus_score": focus_score,
        "focus_label": label
    })

In [ ]:
df_sample = df.sample(300, random_state=42).copy()

In [ ]:
focus_results = df_sample["review"].apply(classify_focus)

df_sample = pd.concat(
    [df_sample.reset_index(drop=True), focus_results.reset_index(drop=True)],
    axis=1
)

In [ ]:
# Semantic score pipline
def get_sentiment_score(text):
    result = sentiment_classifier(text[:512])[0]
    label = result["label"].lower()
    score = result["score"]

    if "negative" in label:
        return -score
    elif "positive" in label:
        return score
    else:
        return 0 

In [ ]:
df_sample["review"] = df_sample["review"].fillna("").astype(str).str.strip()
df_sample["rating"] = pd.to_numeric(df_sample["rating"], errors="coerce")

# drop missing vals
df_sample = df_sample.dropna(subset=["review", "rating"])

# get the sentiment scores for the extracted sample reviews 
df_sample["semantic_sentiment"] = df_sample["review"].apply(get_sentiment_score)

# converts 1-5 rating score to -1 to + 1
df_sample["rating_score"] = (df_sample["rating"] - 3) / 2

df_sample["y_sentiment_score"] = (
    0.7 * df_sample["semantic_sentiment"] +
    0.3 * df_sample["rating_score"]
)

In [ ]:
# Final plotting columns
df_sample = df_sample.copy()

df_sample["x"] = df_sample["focus_score"]
df_sample["y"] = df_sample["y_sentiment_score"]